# Ball and Beam Physics Simulation Experimentation

Simulating a tilting beam balancing a ball with a sine wave control input and live visualization.

## Import Required Libraries

In [ ]:
try:
    from jax import config
    config.update("jax_enable_x64", True)
    import math
    import jax
    import jax.numpy as jnp
    import numpy as np
    %matplotlib inline
    import matplotlib.pyplot as plt

    # local imports
    from rk4 import rk4_step

    # Drake imports for rigid body simulation
    from pydrake.systems.framework import DiagramBuilder
    from pydrake.systems.analysis import Simulator
    from pydrake.geometry import Box, Sphere
    from pydrake.math import RigidTransform
    from pydrake.multibody.plant import AddMultibodyPlantSceneGraph, CoulombFriction
    from pydrake.multibody.tree import SpatialInertia, UnitInertia, RevoluteJoint

    print('Imported packages.')
except Exception as e:
    print('Importing packages failed:')
    print(e)
    raise e

## Define System Constants and Parameters

In [ ]:
# Physical parameters
R = 0.01  # meter (ball radius)
m_ball = 0.05  # kg
m_beam = 0.5  # kg
l_beam = 0.3  # meter
I_ball = 2/5 * m_ball * R * R  # kg m^2
I_beam = m_beam * l_beam * l_beam / 12  # kg m^2
g = 9.81  # m/s^2
D = 100  # friction coefficient at the edges

# Initial conditions
q_init = [-0.1, 0., 0., 0.]  # ball pos[m], ball angle[rad], beam pos[m], beam angle[rad]
qdot_init = [0., 0., 0., 0.]  # ball v[m/s], ball omega[rad/s], beam v(always 0), beam omega[rad/s]

# Simulation parameters
tf = 5  # total simulation time (seconds)
dt = 1e-2  # time step
N = int(tf / dt)  # number of steps

## Implement System Dynamics

The system dynamics function computes state derivatives:

$q=[r, \theta_{ball}, r_{beam}, \theta_{beam}]^T$

$\ddot{q}=\begin{pmatrix}-\frac{5g \sin(\theta_{beam})}{7}  \\ -\frac{5g \sin(\theta_{beam})}{7R} \\ 0 \\ u\end{pmatrix}$

In [ ]:
@jax.jit
def dyn_step(x, u):
    """
    Compute dynamics step.
    
    Input:
        state = x = [q, qdot]
        u -> control input (beam angular acceleration command)
    Output:
        xdot = [qdot, qddot]
    """
    q, qdot = jnp.split(x, 2)
    # Ball dynamics (real-time simulation)
    ball_xddot = -5 / 7 * g * jnp.sin(q[3])
    ball_thetaddot = ball_xddot / R
    qddot = jnp.array([ball_xddot, ball_thetaddot, 0., u])
    return jnp.hstack((qdot, qddot))

## Define Sine Wave Control Input

In [ ]:
# Oscillatory input parameters
# Shared by both JAX and PyDrake implementations.
amplitude = 1  # feedforward sine torque/acceleration magnitude
frequency = 0.7   # Hz
omega = 2 * math.pi * frequency

# Feedback gains to keep motion centered around zero beam angle.
k_theta = 6.0
k_omega = 1.6

def control_input(t, theta_beam, theta_beam_dot):
    """
    Input that produces bounded oscillation around zero.

    u(t) = -k_theta * theta_beam - k_omega * theta_beam_dot + amplitude * sin(omega * t)
    """
    return -k_theta * theta_beam - k_omega * theta_beam_dot + amplitude * jnp.sin(omega * t)

## Run Simulation with Live Plotting

In [ ]:
# Initialize state
x0_jax = jnp.array(q_init + qdot_init)

print('Starting JAX simulation...')
print('Control: u = -k_theta*theta - k_omega*theta_dot + A*sin(omega*t)')
print(f'A={amplitude}, f={frequency} Hz, k_theta={k_theta}, k_omega={k_omega}')

# Storage for results
jax_positions = []
times = []

# Run simulation loop
for k in range(N):
    t = k * dt
    q, qdot = jnp.split(x0_jax, 2)

    # Use the same centered oscillatory input for JAX.
    u = control_input(t, q[3], qdot[3])
    x0_jax = rk4_step(dyn_step, x0_jax, dt, u)

    q_next, _ = jnp.split(x0_jax, 2)
    jax_positions.append(float(q_next[0]))
    times.append(t)

print('JAX simulation complete!')

## PyDrake Implementation for Comparison

Now we implement the same ball and beam system in Drake for side-by-side comparison.

In [ ]:
# Build rigid-body ball-on-beam model in Drake with explicit contact
builder = DiagramBuilder()
plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=1e-3)

instance = plant.AddModelInstance("ball_beam")
beam_thickness = 0.01
beam_width = 0.05
beam_contact_length = 10.0  # long contact track to match 1D JAX assumption
beam_mass_drake = 100.0 * m_beam  # reduce back-reaction while keeping Drake fully independent

beam_inertia = SpatialInertia(
    mass=beam_mass_drake,
    p_PScm_E=np.array([0.0, 0.0, 0.0]),
    G_SP_E=UnitInertia.SolidBox(beam_width, l_beam, beam_thickness),
)
ball_inertia = SpatialInertia(
    mass=m_ball,
    p_PScm_E=np.array([0.0, 0.0, 0.0]),
    G_SP_E=UnitInertia.SolidSphere(R),
)

beam = plant.AddRigidBody("beam", instance, beam_inertia)
ball = plant.AddRigidBody("ball", instance, ball_inertia)

# Beam rotates about x-axis. Ball is free and interacts only through contact.
beam_joint = plant.AddJoint(
    RevoluteJoint(
        "beam_rotation",
        plant.world_frame(),
        beam.body_frame(),
        [1.0, 0.0, 0.0],
        0.0,
    )
)

# Register collision geometries with high friction for rolling without slip.
contact_friction = CoulombFriction(static_friction=1.2, dynamic_friction=1.0)
plant.RegisterCollisionGeometry(
    beam,
    RigidTransform(),
    Box(beam_width, beam_contact_length, beam_thickness),
    "beam_collision",
    contact_friction,
)
plant.RegisterCollisionGeometry(
    ball,
    RigidTransform(),
    Sphere(R),
    "ball_collision",
    contact_friction,
)

# Optional visual geometry for clarity.
plant.RegisterVisualGeometry(
    beam,
    RigidTransform(),
    Box(beam_width, beam_contact_length, beam_thickness),
    "beam_visual",
    np.array([0.35, 0.35, 0.35, 1.0]),
)
plant.RegisterVisualGeometry(
    ball,
    RigidTransform(),
    Sphere(R),
    "ball_visual",
    np.array([0.85, 0.2, 0.2, 1.0]),
)

plant.Finalize()

diagram = builder.Build()
simulator = Simulator(diagram)
root_context = simulator.get_mutable_context()
plant_context = plant.GetMyMutableContextFromRoot(root_context)

# Initial conditions mapped from your state convention
# q_init = [ball_pos, ball_angle, beam_pos, beam_angle]
# qdot_init = [ball_vel, ball_omega, beam_vel, beam_omega]
beam_joint.set_angle(plant_context, q_init[3])
beam_joint.set_angular_rate(plant_context, qdot_init[3])

ball_z0 = 0.5 * beam_thickness + R + 1e-4
X_WBall = RigidTransform(p=np.array([0.0, q_init[0], ball_z0], dtype=np.float64))
plant.SetFreeBodyPose(plant_context, ball, X_WBall)

print('Starting PyDrake rigid body simulation with contact...')

# Storage for Drake results
drake_positions = []
drake_times = []
theta_cmd = q_init[3]
theta_dot_cmd = qdot_init[3]

for k in range(N):
    t = k * dt

    # Prescribed beam kinematics: no dynamic feedback from ball -> beam.
    beam_alpha_cmd = float(control_input(t, theta_cmd, theta_dot_cmd))
    theta_dot_cmd += beam_alpha_cmd * dt
    theta_cmd += theta_dot_cmd * dt
    beam_joint.set_angle(plant_context, theta_cmd)
    beam_joint.set_angular_rate(plant_context, theta_dot_cmd)

    simulator.AdvanceTo(t + dt)

    # Re-apply the commanded beam state to suppress any contact-induced beam drift.
    beam_joint.set_angle(plant_context, theta_cmd)
    beam_joint.set_angular_rate(plant_context, theta_dot_cmd)

    X_WBeam = plant.EvalBodyPoseInWorld(plant_context, beam)
    X_WBall = plant.EvalBodyPoseInWorld(plant_context, ball)
    X_BeamBall = X_WBeam.inverse().multiply(X_WBall)
    drake_positions.append(float(X_BeamBall.translation()[1]))  # ball position in beam frame
    drake_times.append(t)

print('PyDrake rigid body contact simulation complete!')

## Comparison of JAX vs PyDrake

Compare the ball position trajectories from both implementations.

In [ ]:
# Plot both trajectories (sized to fit notebook output)
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)

ax.plot(times, jax_positions, 'b-', linewidth=2.0, label='JAX (RK4)', alpha=0.9)
ax.plot(drake_times, drake_positions, 'r--', linewidth=2.0, label='PyDrake', alpha=0.9)

ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Ball Position (m)', fontsize=11)
ax.set_title('Ball Position: JAX vs PyDrake', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## PyDrake with Physical Beam Length + End Barriers

This variant uses the true beam length (`l_beam`) for contact geometry and adds rigid barriers at both beam ends so the ball stays on the beam.

In [ ]:
# Build a second Drake model with true beam length and side barriers
builder_barrier = DiagramBuilder()
plant_barrier, scene_graph_barrier = AddMultibodyPlantSceneGraph(builder_barrier, time_step=1e-3)

instance_barrier = plant_barrier.AddModelInstance("ball_beam_barrier")
beam_thickness_barrier = beam_thickness
beam_width_barrier = beam_width
beam_contact_length_barrier = l_beam  # use the physical beam length
beam_mass_barrier = beam_mass_drake

beam_inertia_barrier = SpatialInertia(
    mass=beam_mass_barrier,
    p_PScm_E=np.array([0.0, 0.0, 0.0]),
    G_SP_E=UnitInertia.SolidBox(beam_width_barrier, beam_contact_length_barrier, beam_thickness_barrier),
)
ball_inertia_barrier = SpatialInertia(
    mass=m_ball,
    p_PScm_E=np.array([0.0, 0.0, 0.0]),
    G_SP_E=UnitInertia.SolidSphere(R),
)

beam_barrier = plant_barrier.AddRigidBody("beam", instance_barrier, beam_inertia_barrier)
ball_barrier = plant_barrier.AddRigidBody("ball", instance_barrier, ball_inertia_barrier)

beam_joint_barrier = plant_barrier.AddJoint(
    RevoluteJoint(
        "beam_rotation",
        plant_barrier.world_frame(),
        beam_barrier.body_frame(),
        [1.0, 0.0, 0.0],
        0.0,
    )
)

contact_friction_barrier = CoulombFriction(static_friction=1.2, dynamic_friction=1.0)

# Main beam collision geometry with true length.
plant_barrier.RegisterCollisionGeometry(
    beam_barrier,
    RigidTransform(),
    Box(beam_width_barrier, beam_contact_length_barrier, beam_thickness_barrier),
    "beam_collision",
    contact_friction_barrier,
)

# End barriers attached to the beam body to prevent the ball from rolling off.
barrier_thickness = 0.01
barrier_height = 0.05
barrier_y = 0.5 * beam_contact_length_barrier + 0.5 * barrier_thickness
barrier_z = 0.5 * beam_thickness_barrier + 0.5 * barrier_height

plant_barrier.RegisterCollisionGeometry(
    beam_barrier,
    RigidTransform(p=np.array([0.0, barrier_y, barrier_z])),
    Box(beam_width_barrier, barrier_thickness, barrier_height),
    "right_barrier_collision",
    contact_friction_barrier,
)
plant_barrier.RegisterCollisionGeometry(
    beam_barrier,
    RigidTransform(p=np.array([0.0, -barrier_y, barrier_z])),
    Box(beam_width_barrier, barrier_thickness, barrier_height),
    "left_barrier_collision",
    contact_friction_barrier,
)

plant_barrier.RegisterCollisionGeometry(
    ball_barrier,
    RigidTransform(),
    Sphere(R),
    "ball_collision",
    contact_friction_barrier,
)

# Visuals
plant_barrier.RegisterVisualGeometry(
    beam_barrier,
    RigidTransform(),
    Box(beam_width_barrier, beam_contact_length_barrier, beam_thickness_barrier),
    "beam_visual",
    np.array([0.25, 0.35, 0.45, 1.0]),
)
plant_barrier.RegisterVisualGeometry(
    beam_barrier,
    RigidTransform(p=np.array([0.0, barrier_y, barrier_z])),
    Box(beam_width_barrier, barrier_thickness, barrier_height),
    "right_barrier_visual",
    np.array([0.15, 0.15, 0.15, 1.0]),
)
plant_barrier.RegisterVisualGeometry(
    beam_barrier,
    RigidTransform(p=np.array([0.0, -barrier_y, barrier_z])),
    Box(beam_width_barrier, barrier_thickness, barrier_height),
    "left_barrier_visual",
    np.array([0.15, 0.15, 0.15, 1.0]),
)
plant_barrier.RegisterVisualGeometry(
    ball_barrier,
    RigidTransform(),
    Sphere(R),
    "ball_visual",
    np.array([0.9, 0.3, 0.2, 1.0]),
)

plant_barrier.Finalize()

diagram_barrier = builder_barrier.Build()
simulator_barrier = Simulator(diagram_barrier)
root_context_barrier = simulator_barrier.get_mutable_context()
plant_context_barrier = plant_barrier.GetMyMutableContextFromRoot(root_context_barrier)

# Initial conditions.
beam_joint_barrier.set_angle(plant_context_barrier, q_init[3])
beam_joint_barrier.set_angular_rate(plant_context_barrier, qdot_init[3])

ball_z0_barrier = 0.5 * beam_thickness_barrier + R + 1e-4
X_WBall_barrier = RigidTransform(p=np.array([0.0, q_init[0], ball_z0_barrier], dtype=np.float64))
plant_barrier.SetFreeBodyPose(plant_context_barrier, ball_barrier, X_WBall_barrier)

print('Starting PyDrake simulation with physical beam length and end barriers...')

drake_barrier_positions = []
drake_barrier_times = []
theta_cmd_barrier = q_init[3]
theta_dot_cmd_barrier = qdot_init[3]

for k in range(N):
    t = k * dt

    beam_alpha_cmd = float(control_input(t, theta_cmd_barrier, theta_dot_cmd_barrier))
    theta_dot_cmd_barrier += beam_alpha_cmd * dt
    theta_cmd_barrier += theta_dot_cmd_barrier * dt
    beam_joint_barrier.set_angle(plant_context_barrier, theta_cmd_barrier)
    beam_joint_barrier.set_angular_rate(plant_context_barrier, theta_dot_cmd_barrier)

    simulator_barrier.AdvanceTo(t + dt)

    # Keep the beam on the commanded trajectory.
    beam_joint_barrier.set_angle(plant_context_barrier, theta_cmd_barrier)
    beam_joint_barrier.set_angular_rate(plant_context_barrier, theta_dot_cmd_barrier)

    X_WBeam_barrier = plant_barrier.EvalBodyPoseInWorld(plant_context_barrier, beam_barrier)
    X_WBall_barrier = plant_barrier.EvalBodyPoseInWorld(plant_context_barrier, ball_barrier)
    X_BeamBall_barrier = X_WBeam_barrier.inverse().multiply(X_WBall_barrier)
    drake_barrier_positions.append(float(X_BeamBall_barrier.translation()[1]))
    drake_barrier_times.append(t)

print('Barrier-constrained PyDrake simulation complete!')

# Plot and compare all trajectories.
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)
ax.plot(drake_barrier_times, drake_barrier_positions, linewidth=2.0, label='PyDrake (physical beam + barriers)', alpha=0.9)
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Ball Position (m)', fontsize=11)
ax.set_title('Ball Position Comparison', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='best')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()